In [6]:
import pandas as pd
import numpy as np

# สำหรับ IterativeImputer ต้อง enable ก่อน
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.ensemble import RandomForestRegressor


# --- สร้าง DataFrame ตัวอย่าง 10 วัน ---
dates = pd.date_range(start="2025-08-01", periods=10, freq="D")
data = {
    "Date": dates,
    "Showalter": [10, 12, 14, 15, 11, 13, 14, 13, 12, 11],
    "Lifted": [8, 9, 10, 11, 7, 8, 9, 9, 10, 8],
    "SWEAT": [200, 220, 210, 230, 215, 240, 250, 235, 260, 270],
    "dBZ": [35, 40, 42, 45, 38, 41, 42, 41, 39, 37]
}
df = pd.DataFrame(data)

# --- ทำให้ข้อมูลหายไปทั้งแถว 2 วัน (index 2 และ 5) ---
df.loc[[2, 5], df.columns != "Date"] = np.nan

print("ข้อมูลต้นฉบับ (มี missing ทั้งแถว 2 วัน):")
print(df)

# --- เลือกเฉพาะคอลัมน์ตัวเลข (ไม่เอา Date) ---
numeric_cols = df.columns.drop("Date")
df_numeric = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# --- KNN Imputer ---
knn_imputer = KNNImputer(n_neighbors=2)
data_knn_imputed = pd.DataFrame(knn_imputer.fit_transform(df_numeric), columns=numeric_cols)
data_knn_imputed["Date"] = df["Date"].values
data_knn_imputed = data_knn_imputed[["Date"] + numeric_cols.tolist()]

print("\nข้อมูลหลังเติม missing ด้วย KNN:")
print(data_knn_imputed)

# --- Random Forest Imputer ---
rf_estimator = RandomForestRegressor(n_estimators=100, random_state=42)
rf_imputer = IterativeImputer(estimator=rf_estimator, max_iter=10, random_state=42)
data_rf_imputed = pd.DataFrame(rf_imputer.fit_transform(df_numeric), columns=numeric_cols)
data_rf_imputed["Date"] = df["Date"].values
data_rf_imputed = data_rf_imputed[["Date"] + numeric_cols.tolist()]

print("\nข้อมูลหลังเติม missing ด้วย Random Forest:")
print(data_rf_imputed)

# --- Mean Imputer ---
mean_imputer = SimpleImputer(strategy="mean")
data_mean_imputed = pd.DataFrame(mean_imputer.fit_transform(df_numeric), columns=numeric_cols)
data_mean_imputed["Date"] = df["Date"].values
data_mean_imputed = data_mean_imputed[["Date"] + numeric_cols.tolist()]

print("\nข้อมูลหลังเติม missing ด้วย Mean Imputer:")
print(data_mean_imputed)

ข้อมูลต้นฉบับ (มี missing ทั้งแถว 2 วัน):
        Date  Showalter  Lifted  SWEAT   dBZ
0 2025-08-01       10.0     8.0  200.0  35.0
1 2025-08-02       12.0     9.0  220.0  40.0
2 2025-08-03        NaN     NaN    NaN   NaN
3 2025-08-04       15.0    11.0  230.0  45.0
4 2025-08-05       11.0     7.0  215.0  38.0
5 2025-08-06        NaN     NaN    NaN   NaN
6 2025-08-07       14.0     9.0  250.0  42.0
7 2025-08-08       13.0     9.0  235.0  41.0
8 2025-08-09       12.0    10.0  260.0  39.0
9 2025-08-10       11.0     8.0  270.0  37.0

ข้อมูลหลังเติม missing ด้วย KNN:
        Date  Showalter  Lifted  SWEAT     dBZ
0 2025-08-01      10.00   8.000  200.0  35.000
1 2025-08-02      12.00   9.000  220.0  40.000
2 2025-08-03      12.25   8.875  235.0  39.625
3 2025-08-04      15.00  11.000  230.0  45.000
4 2025-08-05      11.00   7.000  215.0  38.000
5 2025-08-06      12.25   8.875  235.0  39.625
6 2025-08-07      14.00   9.000  250.0  42.000
7 2025-08-08      13.00   9.000  235.0  41.000
8 2025

/usr/local/lib/python3.11/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [7]:
import numpy as np

# ตรวจสอบตำแหน่ง missing
missing_positions = np.where(df[numeric_cols].isna())
print("ตำแหน่ง missing:", missing_positions)

# เปรียบเทียบค่าเติม KNN vs RF
filled_diff = data_knn_imputed[numeric_cols] - data_rf_imputed[numeric_cols]
print("ความต่าง KNN-RF (เฉพาะค่าที่เติม):")
print(filled_diff.where(df[numeric_cols].isna()))


ตำแหน่ง missing: (array([2, 2, 2, 2, 5, 5, 5, 5]), array([0, 1, 2, 3, 0, 1, 2, 3]))
ความต่าง KNN-RF (เฉพาะค่าที่เติม):
   Showalter  Lifted  SWEAT    dBZ
0        NaN     NaN    NaN    NaN
1        NaN     NaN    NaN    NaN
2      -0.11  -0.275   7.55 -0.375
3        NaN     NaN    NaN    NaN
4        NaN     NaN    NaN    NaN
5      -0.11  -0.275   7.55 -0.375
6        NaN     NaN    NaN    NaN
7        NaN     NaN    NaN    NaN
8        NaN     NaN    NaN    NaN
9        NaN     NaN    NaN    NaN
